# Unsupervised preprocessing – patient testimonies

This notebook performs **cleaning and basic preprocessing** for the CVRIE unsupervised task.

Goal:
- Start from the provided `Student_Dataset.csv` file (testimonies).
- Extract three logical columns:
  - `id`: numeric identifier of the testimony
  - `color`: optional hidden color code (values such as `0x000000`), **kept only for visualisation later**
  - `text`: original free-text testimony
- Create a cleaned text column that will be used later for **vectorisation and clustering**.

We do **not** create any labels and we **never use the hidden color column for training** – it is only kept so that we can color the clusters for plots during the defense, as requested in the subject.

In [1]:
from pathlib import Path
import csv
import re

import pandas as pd

data_path = Path("Student_Dataset.csv")
data_path

PosixPath('Student_Dataset.csv')

## 1. Load and parse the raw CSV

The file is not a perfect rectangular table:
- some lines are `id, text...`
- other lines are `id, 0xXXXXXX, text...`

We parse it line by line with the `csv` module and manually build three lists: `ids`, `colors`, and `texts`.

In [2]:
HEX_COLOR_RE = re.compile(r"^0x[0-9A-Fa-f]{6}$")

ids = []
colors = []
texts = []

with data_path.open(newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if not row:
            continue

        # Try to read the id (first column)
        try:
            current_id = int(row[0])
        except ValueError:
            # Skip header or malformed line
            continue

        fields = row[1:]
        if not fields:
            continue

        # Detect optional color in second column
        if HEX_COLOR_RE.fullmatch(fields[0]):
            current_color = fields[0]
            text_parts = fields[1:]
        else:
            current_color = None
            text_parts = fields

        text = ",".join(text_parts).strip()

        # Remove surrounding quotes if present
        if len(text) >= 2 and text[0] == text[-1] and text[0] in {'"', "'"}:
            text = text[1:-1].strip()

        if not text:
            continue

        ids.append(current_id)
        colors.append(current_color)
        texts.append(text)

len(ids), len(colors), len(texts)

(1010, 1010, 1010)

## 2. Build a DataFrame and inspect the raw data

In [3]:
df_raw = pd.DataFrame({
    "id": ids,
    "color": colors,
    "text": texts,
})

df_raw.head()

,id,color,text
0,2,0x000000,"The feverish feeling eased, but I’m now breath..."
1,3,0x000000,"While working in the garden, I touched a metal..."
2,4,NaN,A short sprint brought on a twinge in the midd...
3,5,0x000000,I accidentally left my hand on a hot mug for a...
4,6,NaN,A sudden twist while reaching for a bag made o...


We keep the `color` column for later **visualisation only** (to color clusters in plots).
It must **not** be used as input or target during clustering.

## 3. Simple text cleaning

Here we do only basic, transparent operations:
- lowercasing
- normalising some punctuation
- removing unusual characters
- collapsing multiple spaces

In [4]:
def clean_text(text: str) -> str:
    """Very simple text normalisation for patient testimonies."""
    text = text.lower()
    text = (
        text.replace("’", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("–", "-")
        .replace("—", "-")
    )
    # Keep letters, digits, basic punctuation; replace other chars with a space
    text = re.sub(r"[^a-z0-9\s'.,;:?!-]", " ", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_raw["text_clean"] = df_raw["text"].astype(str).map(clean_text)
df_raw[["text", "text_clean"]].head(10)

,text,text_clean
0,"The feverish feeling eased, but I’m now breath...","the feverish feeling eased, but i'm now breath..."
1,"While working in the garden, I touched a metal...","while working in the garden, i touched a metal..."
2,A short sprint brought on a twinge in the midd...,a short sprint brought on a twinge in the midd...
3,I accidentally left my hand on a hot mug for a...,i accidentally left my hand on a hot mug for a...
4,A sudden twist while reaching for a bag made o...,a sudden twist while reaching for a bag made o...
5,"During a birthday party, a hot beverage was kn...","during a birthday party, a hot beverage was kn..."
6,I noticed a rash on my forearms after handling...,i noticed a rash on my forearms after handling...
7,"Since I lost 20 pounds, my headaches have less...","since i lost 20 pounds, my headaches have less..."
8,The rash on my face and jaw is a classic hives...,the rash on my face and jaw is a classic hives...
9,I feel an unexplained pressure behind my eyes ...,i feel an unexplained pressure behind my eyes ...


## 4. Remove empty / very short and duplicate testimonies

We:
- drop rows where the cleaned text is empty
- optionally drop rows where the cleaned text is extremely short (here: < 5 characters)
- drop exact duplicates on the cleaned text

In [5]:
min_length = 5  # you can adjust this if you want to keep everything

df_clean = df_raw.copy()
df_clean = df_clean[df_clean["text_clean"].str.len() > 0]
if min_length > 0:
    df_clean = df_clean[df_clean["text_clean"].str.len() >= min_length]
df_clean = df_clean.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)

df_clean.head()

,id,color,text,text_clean
0,2,0x000000,"The feverish feeling eased, but I’m now breath...","the feverish feeling eased, but i'm now breath..."
1,3,0x000000,"While working in the garden, I touched a metal...","while working in the garden, i touched a metal..."
2,4,NaN,A short sprint brought on a twinge in the midd...,a short sprint brought on a twinge in the midd...
3,5,0x000000,I accidentally left my hand on a hot mug for a...,i accidentally left my hand on a hot mug for a...
4,6,NaN,A sudden twist while reaching for a bag made o...,a sudden twist while reaching for a bag made o...


## 5. Save the cleaned dataset

We save a CSV file that contains:
- `id`
- `color` (for plotting only)
- `text`
- `text_clean` (the column you will feed to TF-IDF / other vectorisation methods)

In [6]:
output_path = Path("cleaned_unsupervised_dataset.csv")
df_clean.to_csv(output_path, index=False)
output_path

PosixPath('cleaned_unsupervised_dataset.csv')